In [15]:
import pandas as pd
import numpy as np
import os, re, gc
import pyarrow.parquet as pq
from collections import defaultdict

pd.options.mode.chained_assignment = None

# ── Chemins ──
DATA_PATH   = 'cert_dataset'
OUTPUT_PATH = 'features_weekly.parquet'

# ── Constantes domaine ──
DOMAIN = '@dtaa.com'
SHORT_SESSION_H  = 0.1
SHORT_DEVICE_H   = 0.1
MIN_HIST_WEEKS   = 3

# ── Mots-clés email suspects ──
SUSPICIOUS_KEYWORDS = [
    'jobsearch','job search','security clearance','contingent upon award',
    'position requires','apply now','bachelor degree','job opening',
    'resignation','two weeks notice','last day','offer letter','new position',
    'keylogger','free keylogger','computer monitoring','monitoring software',
    'spyware','trojan','rootkit','RAT','remote access tool','payload','botnet',
    'do not share','do not forward','internal only','not for distribution',
    'keep this between','off the record',
    'my password is','login credentials','api key','private key','ssh key',
    'access token','secret key',
    'delete after reading','do not tell','just between us','they will never know',
]
SUSPICIOUS_EMAIL_PATTERN = '|'.join(map(re.escape, SUSPICIOUS_KEYWORDS))

# ── Domaines HTTP ──
JOB_DOMAINS = {
    'linkedin.com','indeed.com','monster.com','simplyhired.com',
    'careerbuilder.com','glassdoor.com','job-hunt.org','jobhuntersbible.com',
    'taleo.net','elance.com','fiverr.com','freelancer.com','odesk.com',
    'ziprecruiter.com','dice.com','lever.co','greenhouse.io',
}
SUS_DOMAINS = {
    'wikileaks.org','dropbox.com','mediafire.com','megaupload.com',
    '4shared.com','filesonic.com','fileserve.com','yousendit.com',
    'refog.com','softactivity.com','thepiratebay.org','btjunkie.org',
    'torrentz.eu','demonoid.me','kat.ph','ilivid.com','filestube.com',
    'lockheedmartin.com','northropgrumman.com','boeing.com','raytheon.com',
    'harris.com','megaclick.com','backpage.com',
}
HTTP_WEIGHTS  = {'WWW Visit': 1, 'WWW Download': 2, 'WWW Upload': 4}
HTTP_KEYWORDS = [
    'resume','cv','cover letter','job offer','salary','offer letter',
    'interview','resignation','notice period','reference letter',
    'confidential','internal only','proprietary','trade secret',
    'do not distribute','restricted','source code','database dump',
    'credentials','password','api_key','secret_key','token',
    'drop table','delete all','rm -rf','wipe','disable account',
]
HTTP_KW_PATTERN = re.compile('|'.join(map(re.escape, HTTP_KEYWORDS)), flags=re.IGNORECASE)

# ── Helpers domaine HTTP ──
def _extract_domain(url_series):
    return url_series.str.extract(r'https?://(?:www\.)?([^/?#]+)', expand=False).fillna('')

def _is_job_domain(url_series):
    patt = '|'.join(r'(?:^|\\.)' + re.escape(d) + '$' for d in JOB_DOMAINS)
    return _extract_domain(url_series).str.contains(patt, regex=True, na=False)

def _is_sus_domain(url_series):
    patt = '|'.join(r'(?:^|\\.)' + re.escape(d) + '$' for d in SUS_DOMAINS)
    return _extract_domain(url_series).str.contains(patt, regex=True, na=False)

# ── Extensions fichiers suspects ──
FILE_SUS_EXT = r'\.(exe|bat|ps1|sh|zip|7z|rar|tar|gz|dll|sys)$'

# ── Utilitaires no-leakage ──
def zscore_vs_history(df, user_col, feature_col, min_periods=MIN_HIST_WEEKS):
    """Z-score de la semaine W par rapport à l'historique W-1,W-2,... uniquement."""
    def _zscore_group(series):
        hist      = series.shift(1)
        hist_mean = hist.expanding(min_periods=min_periods).mean()
        hist_std  = hist.expanding(min_periods=min_periods).std().clip(lower=1)
        return (series - hist_mean) / hist_std
    return df.groupby(user_col)[feature_col].transform(_zscore_group).fillna(0)

def cum_sum(df, user_col, col):
    return df.groupby(user_col)[col].transform(lambda x: x.expanding(min_periods=1).sum())

def cum_mean(df, user_col, col):
    return df.groupby(user_col)[col].transform(lambda x: x.expanding(min_periods=1).mean())

def cum_max(df, user_col, col):
    return df.groupby(user_col)[col].transform(lambda x: x.expanding(min_periods=1).max())

def cum_std(df, user_col, col):
    return df.groupby(user_col)[col].transform(lambda x: x.expanding(min_periods=2).std()).fillna(0)

def safe_div(num, den, fill=0.0):
    return np.where(den > 0, num / den, fill)

print('Config OK')

Config OK


In [16]:
# ── Users & grille hebdomadaire ──
users = pd.read_parquet(f'{DATA_PATH}/users.parquet')
users['start_date'] = pd.to_datetime(users['start_date'])
users['end_date']   = pd.to_datetime(users['end_date'])
users.rename(columns={'user_id': 'user'}, inplace=True)

DATASET_END   = pd.Timestamp('2011-06-01')
DATASET_START = pd.Timestamp('2010-01-04')
users['has_left'] = users['end_date'] < DATASET_END
user_end_date = dict(zip(users['user'], users['end_date']))

all_mondays = pd.date_range(start=DATASET_START, end=DATASET_END, freq='W-MON')
user_week_frames = []
for _, row in users.iterrows():
    u_start = max(DATASET_START, row['start_date'].to_period('W').start_time)
    u_end   = min(DATASET_END,   row['end_date'].to_period('W').start_time + pd.Timedelta(weeks=4))
    weeks_i = all_mondays[(all_mondays >= u_start) & (all_mondays <= u_end)]
    user_week_frames.append(pd.DataFrame({'user': row['user'], 'week_start': weeks_i}))

weekly_grid = pd.concat(user_week_frames, ignore_index=True)
weekly_grid = weekly_grid.sort_values(['user','week_start']).reset_index(drop=True)
weekly_grid['week_number'] = weekly_grid.groupby('user').cumcount() + 1

print(f'{len(users)} utilisateurs — Grille : {len(weekly_grid):,} lignes')

4000 utilisateurs — Grille : 284,061 lignes


# Email

In [17]:
# ── Email — Chargement & flags ──
email = pd.read_parquet(f'{DATA_PATH}/email.parquet')
email['date'] = pd.to_datetime(email['date'], format='%m/%d/%Y %H:%M:%S')
email = email.drop(columns=['id']).sort_values('date')

email['is_external']             = (~email['from'].str.contains(DOMAIN, na=False)) | (~email['to'].str.contains(DOMAIN, na=False))
email['has_bcc']                 = email['bcc'].notna() & (email['bcc'] != 'None')
email['has_attachment']          = email['attachments'].notna() & (email['attachments'] != 'None')
email['is_external_with_att']    = email['is_external'] & email['has_attachment']
email['is_after_hours']          = (email['date'].dt.hour > 19) | (email['date'].dt.hour < 7)
email['is_weekend']              = email['date'].dt.dayofweek >= 5
email['is_off_hours_or_weekend'] = email['is_after_hours'] | email['is_weekend']
email['has_suspicious_content']  = email['content'].str.lower().str.contains(SUSPICIOUS_EMAIL_PATTERN, na=False).astype(int)
email['is_sent']                 = (email['activity'] == 'Send').astype(int)
email['week_start']              = email['date'].dt.to_period('W').dt.start_time

# PC principal par user
email_main_pc = (
    email.groupby(['user', 'pc']).size()
    .reset_index(name='cnt').sort_values('cnt', ascending=False)
    .groupby('user').first().reset_index()[['user', 'pc']]
    .rename(columns={'pc': 'email_main_pc'})
)
email = email.merge(email_main_pc, on='user', how='left')
email['is_not_main_pc'] = (email['pc'] != email['email_main_pc']).astype(int)

print(f'Email : {len(email):,} événements')

Email : 10,994,957 événements


In [18]:
# ── Email — Agrégation weekly brute ──
email_w = email.groupby(['user','week_start']).agg(
    nb_emails        = ('date',                       'count'),
    nb_sent          = ('is_sent',                    'sum'),
    nb_external      = ('is_external',                'sum'),
    nb_bcc           = ('has_bcc',                    'sum'),
    nb_ext_att       = ('is_external_with_att',       'sum'),
    nb_off_hrs       = ('is_off_hours_or_weekend',    'sum'),
    nb_suspicious    = ('has_suspicious_content',     'sum'),
    nb_not_main_pc   = ('is_not_main_pc',             'sum'),
).reset_index()

sent_size_w = (
    email[email['activity']=='Send']
    .groupby(['user','week_start'])['size'].mean()
    .reset_index(name='avg_size_sent_w')
)
email_w = email_w.merge(sent_size_w, on=['user','week_start'], how='left')
email_w['avg_size_sent_w'] = email_w['avg_size_sent_w'].fillna(0)

email_w = weekly_grid.merge(email_w, on=['user','week_start'], how='left').fillna(0)
email_w = email_w.sort_values(['user','week_start'])

print(f'Email agrégé : {email_w.shape}')

Email agrégé : (284061, 12)


In [19]:
# ── Email — Features cumulatives (ratios semaines 1..W) ──
for col in ['nb_emails','nb_external','nb_bcc','nb_ext_att','nb_off_hrs','nb_suspicious','nb_not_main_pc','nb_sent']:
    email_w[f'cum_{col}'] = cum_sum(email_w, 'user', col)
email_w['cum_avg_size_sent'] = cum_mean(email_w, 'user', 'avg_size_sent_w')

den = email_w['cum_nb_emails'].clip(lower=1)
email_w['email_external_ratio']                        = email_w['cum_nb_external']    / den
email_w['email_suspicious_content_ratio']              = email_w['cum_nb_suspicious']  / den
email_w['email_bcc_email_ratio']                       = email_w['cum_nb_bcc']         / den
email_w['email_external_email_with_attachment_ratio']   = email_w['cum_nb_ext_att']     / den
email_w['email_after_hours_or_weekend_ratio']           = email_w['cum_nb_off_hrs']     / den
email_w['email_avg_emails_per_week']                    = cum_mean(email_w, 'user', 'nb_emails')
email_w['email_avg_size_sent_email']                    = email_w['cum_avg_size_sent']

# Z-score hebdomadaire brut (pour calculer le max cumulatif)
email_w['_zscore_emails_w'] = zscore_vs_history(email_w, 'user', 'nb_emails')
email_w['email_max_zscore_emails'] = cum_max(email_w, 'user', '_zscore_emails_w')
email_w.loc[email_w['email_max_zscore_emails'] < 0, 'email_max_zscore_emails'] = 0

print('Email cumul OK')

Email cumul OK


In [20]:
# ── Email — Z-score comportemental ──
email_w['email_zscore_last_month'] = zscore_vs_history(email_w, 'user', 'nb_emails')

email_zscore_cols = [
    'email_external_ratio','email_suspicious_content_ratio','email_bcc_email_ratio',
    'email_external_email_with_attachment_ratio','email_after_hours_or_weekend_ratio',
    'email_avg_emails_per_week','email_avg_size_sent_email',
]
for c in email_zscore_cols:
    email_w[f'z_{c}'] = zscore_vs_history(email_w, 'user', c)

print('Email z-scores OK')

Email z-scores OK


# File

In [ ]:
# ── File — Chargement & flags ──
file = pd.read_parquet(f'{DATA_PATH}/file.parquet')
file['date'] = pd.to_datetime(file['date'], format='%m/%d/%Y %H:%M:%S')
file = file.drop(columns=['id']).sort_values('date')

file['is_copy']      = (file['activity'] == 'Copy').astype(int)
file['is_write']     = (file['activity'] == 'Write').astype(int)
file['is_delete']    = (file['activity'] == 'Delete').astype(int)
file['is_open']      = (file['activity'] == 'Open').astype(int)
file['copy_to_rem']  = ((file['activity']=='Copy') & file['to_removable_media'].fillna(False)).astype(int)
file['from_rem']     = ((file['activity'].isin(['Copy','Open'])) & file['from_removable_media'].fillna(False)).astype(int)
file['is_off_hours'] = ((file['date'].dt.hour > 19) | (file['date'].dt.hour < 7) | (file['date'].dt.dayofweek >= 5)).astype(int)
file['is_suspicious'] = file['filename'].str.lower().str.contains(FILE_SUS_EXT, na=False).astype(int)

file['prev_act']  = file.groupby(['user','filename'])['activity'].shift(1)
file['open_then_copy']    = ((file['activity']=='Copy')   & (file['prev_act']=='Open')).astype(int)
file['copy_then_delete']  = ((file['activity']=='Delete') & (file['prev_act']=='Copy')).astype(int)

file['week_start'] = file['date'].dt.to_period('W').dt.start_time

# Decoy
decoy = pd.read_parquet(f'{DATA_PATH}/decoy_file.parquet')
decoy_filenames = set(decoy['decoy_filename'].str.lower())
file['is_decoy'] = file['filename'].str.lower().isin(decoy_filenames).astype(int)

# PC principal
file_main_pc = (
    file.groupby(['user','pc']).size().reset_index(name='cnt').sort_values('cnt',ascending=False)
    .groupby('user').first().reset_index()[['user','pc']].rename(columns={'pc':'file_main_pc'})
)
file = file.merge(file_main_pc, on='user', how='left')
file['is_not_main_pc'] = (file['pc'] != file['file_main_pc']).astype(int)

print(f'File : {len(file):,} événements')

C:\Users\OsakaGamingMaroc\AppData\Local\Temp\ipykernel_15864\2670283872.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  file['is_suspicious'] = file['filename'].str.lower().str.contains(FILE_SUS_EXT, na=False).astype(int)


File : 2,014,883 événements


In [22]:
# ── File — Agrégation weekly brute ──
file_w = file.groupby(['user','week_start']).agg(
    nb_events          = ('date',               'count'),
    nb_copy            = ('is_copy',            'sum'),
    nb_write           = ('is_write',           'sum'),
    nb_delete          = ('is_delete',          'sum'),
    nb_copy_to_rem     = ('copy_to_rem',        'sum'),
    nb_from_rem        = ('from_rem',           'sum'),
    nb_unique_files    = ('filename',           'nunique'),
    nb_off_hrs         = ('is_off_hours',       'sum'),
    nb_suspicious      = ('is_suspicious',      'sum'),
    nb_open_then_copy  = ('open_then_copy',     'sum'),
    nb_copy_del        = ('copy_then_delete',   'sum'),
    nb_decoy           = ('is_decoy',           'sum'),
    nb_not_main_pc     = ('is_not_main_pc',     'sum'),
).reset_index()

file_w = weekly_grid.merge(file_w, on=['user','week_start'], how='left').fillna(0)
file_w = file_w.sort_values(['user','week_start'])

print(f'File agrégé : {file_w.shape}')

File agrégé : (284061, 16)


In [23]:
# ── File — Features cumulatives ──
for col in ['nb_events','nb_copy','nb_write','nb_delete','nb_copy_to_rem','nb_from_rem',
            'nb_unique_files','nb_off_hrs','nb_suspicious','nb_open_then_copy','nb_copy_del',
            'nb_decoy','nb_not_main_pc']:
    file_w[f'cum_{col}'] = cum_sum(file_w, 'user', col)

den_ev = file_w['cum_nb_events'].clip(lower=1)
den_cp = file_w['cum_nb_copy'].clip(lower=1)

file_w['file_copy_ratio']                  = file_w['cum_nb_copy']           / den_ev
file_w['file_write_ratio']                 = file_w['cum_nb_write']          / den_ev
file_w['file_delete_ratio']                = file_w['cum_nb_delete']         / den_ev
file_w['file_copy_to_removable_ratio']     = file_w['cum_nb_copy_to_rem']   / den_cp
file_w['file_from_removable_ratio']        = file_w['cum_nb_from_rem']      / den_ev
file_w['file_events_per_file']             = safe_div(file_w['cum_nb_events'].values, file_w['cum_nb_unique_files'].clip(lower=1).values)
file_w['file_off_hours_ratio']             = file_w['cum_nb_off_hrs']       / den_ev
file_w['file_suspicious_file_content_ratio'] = file_w['cum_nb_suspicious']  / den_ev
file_w['file_open_then_copy_ratio']        = file_w['cum_nb_open_then_copy']/ den_cp
file_w['file_copy_then_delete_ratio']      = file_w['cum_nb_copy_del']      / den_cp
file_w['decoy_file_nb_access_decoy_file']  = file_w['cum_nb_decoy']

# Z-score file activity -> max cumulatif
file_w['_zscore_file_w'] = zscore_vs_history(file_w, 'user', 'nb_events')
file_w['file_max_zscore_file_activity'] = cum_max(file_w, 'user', '_zscore_file_w')
file_w.loc[file_w['file_max_zscore_file_activity'] < 0, 'file_max_zscore_file_activity'] = 0

print('File cumul OK')

File cumul OK


In [24]:
# ── File — Z-scores comportementaux ──
file_w['file_zscore_last_month'] = zscore_vs_history(file_w, 'user', 'nb_events')

file_zscore_cols = [
    'file_copy_ratio','file_write_ratio','file_delete_ratio',
    'file_copy_to_removable_ratio','file_from_removable_ratio',
    'file_events_per_file','file_off_hours_ratio','file_suspicious_file_content_ratio',
    'file_open_then_copy_ratio','file_copy_then_delete_ratio',
]
for c in file_zscore_cols:
    file_w[f'z_{c}'] = zscore_vs_history(file_w, 'user', c)

print('File z-scores OK')

File z-scores OK


# Logon

In [25]:
# ── Logon — Chargement, sessions, flags ──
logon = pd.read_parquet(f'{DATA_PATH}/logon.parquet')
logon['date'] = pd.to_datetime(logon['date'], format='%m/%d/%Y %H:%M:%S')
logon = logon.drop(columns=['id']).sort_values('date')
logon['is_after_hours'] = (logon['date'].dt.hour > 19) | (logon['date'].dt.hour < 7)
logon['week_start']     = logon['date'].dt.to_period('W').dt.start_time

# PC principal
logon_main_pc = (
    logon[logon['activity']=='Logon']
    .groupby(['user','pc']).size()
    .reset_index(name='cnt').sort_values('cnt',ascending=False)
    .groupby('user').first().reset_index()[['user','pc']]
    .rename(columns={'pc':'logon_main_pc'})
)
logon = logon.merge(logon_main_pc, on='user', how='left')
logon['is_not_main_pc'] = (logon['pc'] != logon['logon_main_pc']).astype(int)

# Sessions
logon_ev  = logon[logon['activity']=='Logon'].copy()
logoff_ev = logon[logon['activity']=='Logoff'].copy()
logon_ev['rank']  = logon_ev.groupby('user').cumcount()
logoff_ev['rank'] = logoff_ev.groupby('user').cumcount()

sessions = pd.merge(
    logon_ev[['user','rank','pc','date','week_start','is_after_hours']],
    logoff_ev[['user','rank','date']].rename(columns={'date':'date_end'}),
    on=['user','rank'], how='inner'
)
sessions.rename(columns={'date': 'date_start'}, inplace=True)
sessions['duration_h'] = (sessions['date_end'] - sessions['date_start']).dt.total_seconds() / 3600
sessions = sessions[(sessions['duration_h'] > 0) & (sessions['duration_h'] < 24)]
sessions['is_short'] = (sessions['duration_h'] < SHORT_SESSION_H).astype(int)

# Multi-PC simultané
logon_ev['prev_pc'] = logon_ev.groupby('user')['pc'].shift(1)
logon_ev['multi_pc_flag'] = (
    (logon_ev['pc'] != logon_ev['prev_pc']) & logon_ev['prev_pc'].notna()
).astype(int)

print(f'Logon : {len(logon):,} | Sessions : {len(sessions):,}')

Logon : 3,530,285 | Sessions : 999,050


In [26]:
# ── Logon — Agrégation weekly brute ──
logon_w = logon.groupby(['user','week_start']).agg(
    nb_logon       = ('activity', lambda x: (x=='Logon').sum()),
    nb_logoff      = ('activity', lambda x: (x=='Logoff').sum()),
    nb_after_hrs   = ('is_after_hours', 'sum'),
    nb_not_main_pc = ('is_not_main_pc', 'sum'),
).reset_index()

multi_pc_w = logon_ev.groupby(['user','week_start'])['multi_pc_flag'].sum().reset_index(name='nb_multi_pc')
logon_w = logon_w.merge(multi_pc_w, on=['user','week_start'], how='left')

sessions_w = sessions.groupby(['user','week_start']).agg(
    session_dur_avg_w = ('duration_h', 'mean'),
    session_dur_std_w = ('duration_h', 'std'),
    session_nb        = ('duration_h', 'count'),
    session_nb_short  = ('is_short',   'sum'),
).reset_index()
logon_w = logon_w.merge(sessions_w, on=['user','week_start'], how='left')

# Sessions after-hours
sess_ah_w = sessions.groupby(['user','week_start'])['is_after_hours'].sum().reset_index(name='nb_session_after_hrs')
logon_w = logon_w.merge(sess_ah_w, on=['user','week_start'], how='left')

logon_w = weekly_grid.merge(logon_w, on=['user','week_start'], how='left').fillna(0)
logon_w = logon_w.sort_values(['user','week_start'])

# Unclosed sessions this week
logon_w['nb_unclosed_w'] = (logon_w['nb_logon'] - logon_w['nb_logoff']).clip(lower=0)

print(f'Logon agrégé : {logon_w.shape}')

Logon agrégé : (284061, 14)


In [27]:
# ── Logon — Features cumulatives ──
for col in ['nb_logon','nb_unclosed_w','nb_after_hrs','session_nb','session_nb_short','nb_multi_pc','nb_not_main_pc','nb_session_after_hrs']:
    logon_w[f'cum_{col}'] = cum_sum(logon_w, 'user', col)

logon_w['logon_average_session_duration']     = cum_mean(logon_w, 'user', 'session_dur_avg_w')
logon_w['logon_session_duration_variability']  = cum_max(logon_w, 'user', 'session_dur_std_w')

den_logon = logon_w['cum_nb_logon'].clip(lower=1)
den_sess  = logon_w['cum_session_nb'].clip(lower=1)

logon_w['logon_ratio_short_sessions']          = safe_div(logon_w['cum_session_nb_short'].values, den_sess.values)
logon_w['logon_used_multi_pc_simultaneously']   = (logon_w['cum_nb_multi_pc'] > 0).astype(int)
logon_w['logon_unclosed_session_ratio']         = safe_div(logon_w['cum_nb_unclosed_w'].values, den_logon.values)
logon_w['logon_after_hours_logon_ratio']        = safe_div(logon_w['cum_nb_after_hrs'].values, den_logon.values)

# Max z-score logon (cumulatif)
logon_w['_zscore_logon_w'] = zscore_vs_history(logon_w, 'user', 'nb_logon')
logon_w['logon_max_zscore_logon'] = cum_max(logon_w, 'user', '_zscore_logon_w')
logon_w.loc[logon_w['logon_max_zscore_logon'] < 0, 'logon_max_zscore_logon'] = 0

# Max z-score logon after hours
logon_w['_zscore_logon_ah_w'] = zscore_vs_history(logon_w, 'user', 'nb_session_after_hrs')
logon_w['logon_max_zscore_logon_after_hours'] = cum_max(logon_w, 'user', '_zscore_logon_ah_w')
logon_w.loc[logon_w['logon_max_zscore_logon_after_hours'] < 0, 'logon_max_zscore_logon_after_hours'] = 0

print('Logon cumul OK')

Logon cumul OK


In [28]:
# ── Logon — Z-scores comportementaux ──
logon_w['logon_zscore_last_month'] = zscore_vs_history(logon_w, 'user', 'nb_logon')

logon_zscore_cols = [
    'logon_average_session_duration','logon_ratio_short_sessions',
    'logon_unclosed_session_ratio','logon_after_hours_logon_ratio',
]
for c in logon_zscore_cols:
    logon_w[f'z_{c}'] = zscore_vs_history(logon_w, 'user', c)

print('Logon z-scores OK')

Logon z-scores OK


# Device

In [29]:
# ── Device — Chargement & sessions ──
device = pd.read_parquet(f'{DATA_PATH}/device.parquet')
device['date'] = pd.to_datetime(device['date'], format='%m/%d/%Y %H:%M:%S')
device = device.drop(columns=['id']).sort_values('date')
device['is_after_hours'] = ((device['date'].dt.hour > 19) | (device['date'].dt.hour < 7)).astype(int)
device['week_start']     = device['date'].dt.to_period('W').dt.start_time

device_con = device[device['activity'] == 'Connect'].copy()
device_dis = device[device['activity'] == 'Disconnect'].copy()
device_con['rank'] = device_con.groupby('user').cumcount()
device_dis['rank'] = device_dis.groupby('user').cumcount()

device_sessions = pd.merge(
    device_con[['user','rank','pc','date','week_start','is_after_hours']],
    device_dis[['user','rank','date']].rename(columns={'date':'date_end'}),
    on=['user','rank'], how='inner'
)
device_sessions.rename(columns={'date':'date_start'}, inplace=True)
device_sessions['duration_h'] = (device_sessions['date_end'] - device_sessions['date_start']).dt.total_seconds() / 3600
device_sessions = device_sessions[(device_sessions['duration_h'] >= 0) & (device_sessions['duration_h'] < 48)]
device_sessions['is_short'] = (device_sessions['duration_h'] < SHORT_DEVICE_H).astype(int)

# PC principal device
device_main_pc = (
    device_con.groupby(['user','pc']).size().reset_index(name='cnt').sort_values('cnt',ascending=False)
    .groupby('user').first().reset_index()[['user','pc']].rename(columns={'pc':'device_main_pc'})
)
device_con = device_con.merge(device_main_pc, on='user', how='left')
device_con['is_not_main_pc'] = (device_con['pc'] != device_con['device_main_pc']).astype(int)

print(f'Device : {len(device_con):,} connexions | Sessions : {len(device_sessions):,}')

Device : 778,354 connexions | Sessions : 562,413


In [30]:
# ── Device — Agrégation weekly brute ──
device_w = device_con.groupby(['user','week_start']).agg(
    nb_conn        = ('date',            'count'),
    nb_after_hrs   = ('is_after_hours',  'sum'),
    nb_not_main_pc = ('is_not_main_pc',  'sum'),
).reset_index()

dev_sess_w = device_sessions.groupby(['user','week_start']).agg(
    con_dur_avg_w    = ('duration_h', 'mean'),
    con_dur_std_w    = ('duration_h', 'std'),
    nb_short_conn    = ('is_short',    'sum'),
    nb_sess          = ('duration_h',  'count'),
).reset_index()
device_w = device_w.merge(dev_sess_w, on=['user','week_start'], how='left')

device_w = weekly_grid.merge(device_w, on=['user','week_start'], how='left').fillna(0)
device_w = device_w.sort_values(['user','week_start'])

print(f'Device agrégé : {device_w.shape}')

Device agrégé : (284061, 10)


In [31]:
# ── Device — Features cumulatives ──
for col in ['nb_conn','nb_after_hrs','nb_short_conn','nb_sess','nb_not_main_pc']:
    device_w[f'cum_{col}'] = cum_sum(device_w, 'user', col)

device_w['device_con_duration_avg']                   = cum_mean(device_w, 'user', 'con_dur_avg_w')
device_w['device_con_duration_std']                   = cum_std(device_w, 'user', 'con_dur_avg_w')
device_w['device_ratio_short_device_conn_interval']   = safe_div(device_w['cum_nb_short_conn'].values, device_w['cum_nb_sess'].clip(lower=1).values)
device_w['device_avg_device_conn_per_week']           = cum_mean(device_w, 'user', 'nb_conn')
device_w['device_ratio_device_conn_after_hours']      = safe_div(device_w['cum_nb_after_hrs'].values, device_w['cum_nb_conn'].clip(lower=1).values)

# Max z-score device
device_w['_zscore_device_w'] = zscore_vs_history(device_w, 'user', 'nb_conn')
device_w['device_max_zscore_device_week'] = cum_max(device_w, 'user', '_zscore_device_w')
device_w.loc[device_w['device_max_zscore_device_week'] < 0, 'device_max_zscore_device_week'] = 0

print('Device cumul OK')

Device cumul OK


In [32]:
# ── Device — Z-scores comportementaux ──
device_w['device_zscore_last_month'] = zscore_vs_history(device_w, 'user', 'nb_conn')

device_zscore_cols = [
    'device_con_duration_avg','device_con_duration_std',
    'device_ratio_short_device_conn_interval','device_avg_device_conn_per_week',
    'device_ratio_device_conn_after_hours',
]
for c in device_zscore_cols:
    device_w[f'z_{c}'] = zscore_vs_history(device_w, 'user', c)

print('Device z-scores OK')

Device z-scores OK


# HTTP

In [33]:
# ── HTTP — Chargement par chunks ──
http_chunks    = []
http_dom_chunks = []

pf = pq.ParquetFile(f'{DATA_PATH}/http.parquet')

for i, batch in enumerate(pf.iter_batches(batch_size=1_000_000)):
    chunk = batch.to_pandas()
    chunk['activity']    = chunk['activity'].astype(str)
    chunk['datetime']    = pd.to_datetime(chunk['date'])
    chunk['week_start']  = chunk['datetime'].dt.to_period('W').dt.start_time
    chunk['hour']        = chunk['datetime'].dt.hour
    chunk['dow']         = chunk['datetime'].dt.dayofweek
    chunk['day']         = chunk['datetime'].dt.date

    chunk['is_off_hrs']  = ((chunk['dow'] < 5) & ((chunk['hour'] < 7) | (chunk['hour'] >= 19))).astype(int)
    chunk['is_weekend']  = (chunk['dow'] >= 5).astype(int)
    chunk['is_off_or_wk']= (chunk['is_off_hrs'] | chunk['is_weekend']).astype(int)
    chunk['weight']      = chunk['activity'].map(HTTP_WEIGHTS).fillna(0)
    chunk['is_upload']   = (chunk['activity'] == 'WWW Upload').astype(int)
    chunk['is_download'] = (chunk['activity'] == 'WWW Download').astype(int)
    chunk['is_job']      = _is_job_domain(chunk['url']).astype(int)
    chunk['is_sus']      = _is_sus_domain(chunk['url']).astype(int)
    chunk['_domain']     = _extract_domain(chunk['url'])

    transfer_mask = chunk['activity'].isin(['WWW Upload','WWW Download'])
    chunk.loc[transfer_mask, 'kw_flag'] = (
        chunk.loc[transfer_mask,'content'].fillna('').str.count(HTTP_KW_PATTERN)
    )
    chunk['kw_flag'] = chunk['kw_flag'].fillna(0).astype(int)

    agg = chunk.groupby(['user','week_start']).agg(
        nb_total       = ('datetime',    'count'),
        nb_upload      = ('is_upload',   'sum'),
        nb_download    = ('is_download', 'sum'),
        nb_off_or_wk   = ('is_off_or_wk','sum'),
        nb_kw_flag     = ('kw_flag',     'sum'),
        nb_sus_domain  = ('is_sus',      'sum'),
        nb_job_domain  = ('is_job',      'sum'),
        nb_days_active = ('day',         'nunique'),
    ).reset_index()
    http_chunks.append(agg)

    dom_chunk = chunk[['user','week_start','_domain']].drop_duplicates()
    http_dom_chunks.append(dom_chunk)

    del chunk, agg, dom_chunk, transfer_mask
    gc.collect()
    if i % 10 == 0:
        print(f'Batch {i}')

print('Agrégation finale HTTP...')
http_raw = (
    pd.concat(http_chunks, ignore_index=True)
    .groupby(['user','week_start'])
    .agg({
        'nb_total':       'sum',
        'nb_upload':      'sum',
        'nb_download':    'sum',
        'nb_off_or_wk':   'sum',
        'nb_kw_flag':     'sum',
        'nb_sus_domain':  'sum',
        'nb_job_domain':  'sum',
        'nb_days_active': 'sum',
    })
    .reset_index()
)

http_unique_domains = (
    pd.concat(http_dom_chunks, ignore_index=True)
    .drop_duplicates(subset=['user','week_start','_domain'])
    .groupby(['user','week_start'])['_domain'].nunique()
    .reset_index(name='nb_unique_domains')
)
http_raw = http_raw.merge(http_unique_domains, on=['user','week_start'], how='left')

del http_chunks, http_dom_chunks
gc.collect()
print(f'HTTP brut : {len(http_raw):,} paires')

Batch 0
Batch 10
Batch 20
Batch 30
Batch 40
Batch 50
Batch 60
Batch 70
Batch 80
Batch 90
Batch 100
Batch 110
Agrégation finale HTTP...
HTTP brut : 283,518 paires


In [34]:
# ── HTTP — Agrégation weekly + feature suspecte fusionnée ──
http_w = weekly_grid.merge(http_raw, on=['user','week_start'], how='left').fillna(0)
http_w = http_w.sort_values(['user','week_start'])

# Feature locale : ratio actions suspectes semaine W (job_domain + sus_domain + keyword)
n_h = http_w['nb_total'].clip(lower=1)
http_w['http_suspicious_ratio'] = (http_w['nb_job_domain'] + http_w['nb_sus_domain'] + http_w['nb_kw_flag']) / n_h

print(f'HTTP agrégé : {http_w.shape}')

HTTP agrégé : (284061, 13)


In [35]:
# ── HTTP — Features cumulatives ──
for col in ['nb_total','nb_upload','nb_download','nb_off_or_wk','nb_kw_flag',
            'nb_unique_domains','nb_days_active']:
    http_w[f'cum_{col}'] = cum_sum(http_w, 'user', col)

den_h = http_w['cum_nb_total'].clip(lower=1)

http_w['http_avg_requests_per_day']  = safe_div(http_w['cum_nb_total'].values, http_w['cum_nb_days_active'].clip(lower=1).values)
http_w['http_total_keyword_flags']   = http_w['cum_nb_kw_flag']
http_w['http_upload_ratio']          = http_w['cum_nb_upload']    / den_h
http_w['http_download_ratio']        = http_w['cum_nb_download']  / den_h
http_w['http_off_hours_ratio']       = http_w['cum_nb_off_or_wk'] / den_h
http_w['http_unique_domains']        = http_w['cum_nb_unique_domains']

# Max z-score HTTP
http_w['_zscore_http_w'] = zscore_vs_history(http_w, 'user', 'nb_total')
http_w['http_max_zscore_activity'] = cum_max(http_w, 'user', '_zscore_http_w')
http_w.loc[http_w['http_max_zscore_activity'] < 0, 'http_max_zscore_activity'] = 0

print('HTTP cumul OK')

HTTP cumul OK


In [36]:
# ── HTTP — Z-scores comportementaux ──
http_w['http_last_month_zscore'] = zscore_vs_history(http_w, 'user', 'nb_total')

http_zscore_cols = [
    'http_avg_requests_per_day','http_upload_ratio','http_download_ratio',
    'http_off_hours_ratio','http_suspicious_ratio',
]
for c in http_zscore_cols:
    http_w[f'z_{c}'] = zscore_vs_history(http_w, 'user', c)

print('HTTP z-scores OK')

HTTP z-scores OK


# Psycho & LDAP

In [37]:
# ── Psychométrique (OCEAN) ──
psycho = pd.read_parquet(f'{DATA_PATH}/psychometric.parquet')
psycho.rename(columns={'user_id':'user'}, inplace=True)
psycho.drop(columns=['employee_name'], inplace=True, errors='ignore')

for trait in ['O','C','E','A','N']:
    col = psycho[trait]
    psycho[f'{trait}_norm'] = (col - col.min()) / (col.max() - col.min())

psycho['psycho_ocean_risk_score'] = (
    psycho['N_norm'] *
    (1 - psycho['C_norm']) *
    (1 - psycho['A_norm']) *
    (1 - psycho['E_norm']) *
    psycho['O_norm']
)
psycho_feat = psycho[['user','psycho_ocean_risk_score']]

print('Psycho OK')

Psycho OK


In [38]:
# ── LDAP ──
ldap_path = f'{DATA_PATH}/LDAP'
ldap_all = []
for f_name in sorted(os.listdir(ldap_path)):
    df_tmp = pd.read_parquet(f'{ldap_path}/{f_name}')
    df_tmp['month'] = f_name.replace('.parquet','')
    ldap_all.append(df_tmp)
ldap = pd.concat(ldap_all, ignore_index=True)
ldap['month'] = pd.to_datetime(ldap['month'], errors='coerce')

user_tenure = ldap.groupby('user_id').agg(
    ldap_start = ('month','min'),
    ldap_end   = ('month','max')
).reset_index()
user_tenure['ldap_tenure_months'] = (
    (user_tenure['ldap_end'] - user_tenure['ldap_start']).dt.days / 30
).round(1)

mu, sigma = user_tenure['ldap_tenure_months'].mean(), user_tenure['ldap_tenure_months'].std()
user_tenure['ldap_zscore_departure'] = (user_tenure['ldap_tenure_months'] - mu) / max(sigma, 1)
user_tenure.rename(columns={'user_id':'user'}, inplace=True)
ldap_feat = user_tenure[['user','ldap_zscore_departure']]

print('LDAP OK')

LDAP OK


# Assemblage final

In [39]:
# ── Colonnes à conserver par source ──
EMAIL_FEAT = [
    'email_external_ratio','email_suspicious_content_ratio','email_bcc_email_ratio',
    'email_external_email_with_attachment_ratio','email_after_hours_or_weekend_ratio',
    'email_max_zscore_emails','email_avg_emails_per_week','email_avg_size_sent_email',
    'email_zscore_last_month',
] + [c for c in email_w.columns if c.startswith('z_email')]

FILE_FEAT = [
    'file_copy_ratio','file_write_ratio','file_delete_ratio',
    'file_copy_to_removable_ratio','file_from_removable_ratio',
    'file_events_per_file','file_off_hours_ratio','file_suspicious_file_content_ratio',
    'file_open_then_copy_ratio','file_copy_then_delete_ratio',
    'file_max_zscore_file_activity','file_zscore_last_month',
    'decoy_file_nb_access_decoy_file',
] + [c for c in file_w.columns if c.startswith('z_file')]

LOGON_FEAT = [
    'logon_average_session_duration','logon_session_duration_variability',
    'logon_ratio_short_sessions','logon_used_multi_pc_simultaneously',
    'logon_unclosed_session_ratio','logon_after_hours_logon_ratio',
    'logon_max_zscore_logon','logon_max_zscore_logon_after_hours',
    'logon_zscore_last_month',
] + [c for c in logon_w.columns if c.startswith('z_logon')]

DEVICE_FEAT = [
    'device_con_duration_avg','device_con_duration_std',
    'device_ratio_short_device_conn_interval','device_avg_device_conn_per_week',
    'device_max_zscore_device_week','device_ratio_device_conn_after_hours',
    'device_zscore_last_month',
] + [c for c in device_w.columns if c.startswith('z_device')]

HTTP_FEAT = [
    'http_avg_requests_per_day','http_total_keyword_flags',
    'http_upload_ratio','http_download_ratio','http_off_hours_ratio',
    'http_last_month_zscore','http_max_zscore_activity',
    'http_suspicious_ratio','http_unique_domains',
] + [c for c in http_w.columns if c.startswith('z_http')]

# ── Merge séquentiel ──
feat = weekly_grid[['user','week_start','week_number']].copy()

def _merge(base, df, cols, on=['user','week_start']):
    keep = on + [c for c in cols if c in df.columns]
    return base.merge(df[keep], on=on, how='left')

feat = _merge(feat, email_w,  EMAIL_FEAT)
feat = _merge(feat, file_w,   FILE_FEAT)
feat = _merge(feat, logon_w,  LOGON_FEAT)
feat = _merge(feat, device_w, DEVICE_FEAT)
feat = _merge(feat, http_w,   HTTP_FEAT)
feat = feat.merge(psycho_feat, on='user', how='left')
feat = feat.merge(ldap_feat, on='user', how='left')

# ── Features cross-source ──
# Activité hors PC principal (moyenne des ratios cumulatifs)
not_main_cols = []
for src, src_w in [('email', email_w), ('logon', logon_w), ('file', file_w), ('device', device_w)]:
    col_name = f'_{src}_not_main_pc_ratio'
    ratio_col = [c for c in src_w.columns if 'not_main_pc' in c and 'cum' in c]
    if ratio_col:
        feat = feat.merge(
            src_w[['user','week_start',ratio_col[0]]].rename(columns={ratio_col[0]: col_name}),
            on=['user','week_start'], how='left'
        )
        feat[col_name] = feat[col_name].fillna(0)
        not_main_cols.append(col_name)

if not_main_cols:
    feat['users_activity_not_main_pc_ratio'] = feat[not_main_cols].mean(axis=1)
    feat.drop(columns=not_main_cols, inplace=True)
else:
    feat['users_activity_not_main_pc_ratio'] = 0.0

# Post-departure activity
end_dates_s = feat['user'].map(user_end_date)
feat['_weeks_to_dep'] = ((end_dates_s - feat['week_start']).dt.days / 7).round(1)
feat['_is_post_dep']  = (feat['_weeks_to_dep'] < 0).astype(int)

# Activité totale par semaine
for src_name, src_w, nb_col in [
    ('email', email_w, 'nb_emails'), ('logon', logon_w, 'nb_logon'),
    ('file', file_w, 'nb_events'), ('device', device_w, 'nb_conn'),
    ('http', http_w, 'nb_total'),
]:
    feat = feat.merge(
        src_w[['user','week_start',nb_col]].rename(columns={nb_col: f'_nb_{src_name}'}),
        on=['user','week_start'], how='left'
    )

nb_cols_tmp = [c for c in feat.columns if c.startswith('_nb_')]
feat['_total_activity'] = feat[nb_cols_tmp].fillna(0).sum(axis=1)
feat['users_post_departure_activity'] = feat['_is_post_dep'] * feat['_total_activity']

# Pre-departure ratio
feat['users_pre_departure_ratio'] = np.where(
    (feat['_weeks_to_dep'] >= 0) & (feat['_weeks_to_dep'] <= 4),
    safe_div(feat['_total_activity'].values,
             feat.groupby('user')['_total_activity'].transform('mean').clip(lower=1).values),
    0
)

# Nettoyage colonnes temporaires
tmp_cols = [c for c in feat.columns if c.startswith('_')]
feat.drop(columns=tmp_cols, inplace=True)

feat.fillna(0, inplace=True)
feat.replace([np.inf, -np.inf], 0, inplace=True)
feat = feat.sort_values(['user','week_start']).reset_index(drop=True)

print(f'Dataset final : {feat.shape}  ({feat.shape[1]} colonnes)')
feat.head()

Dataset final : (284061, 86)  (86 colonnes)


,user,week_start,week_number,email_external_ratio,email_suspicious_content_ratio,email_bcc_email_ratio,email_external_email_with_attachment_ratio,email_after_hours_or_weekend_ratio,email_max_zscore_emails,email_avg_emails_per_week,...,z_http_avg_requests_per_day,z_http_upload_ratio,z_http_download_ratio,z_http_off_hours_ratio,z_http_suspicious_ratio,psycho_ocean_risk_score,ldap_zscore_departure,users_activity_not_main_pc_ratio,users_post_departure_activity,users_pre_departure_ratio
0,AAB0162,2010-01-04,1,0.222222,0.000000,0.0,0.000000,0.0,0.0,45.00,...,0.000000,0.0,0.0,0.0,0.000000,0.062937,0.27574,0.0,0.0,0.0
1,AAB0162,2010-01-11,2,0.166667,0.000000,0.0,0.011111,0.0,0.0,45.00,...,0.000000,0.0,0.0,0.0,0.000000,0.062937,0.27574,0.0,0.0,0.0
2,AAB0162,2010-01-18,3,0.229630,0.000000,0.0,0.022222,0.0,0.0,45.00,...,0.000000,0.0,0.0,0.0,0.000000,0.062937,0.27574,0.0,0.0,0.0
3,AAB0162,2010-01-25,4,0.216374,0.000000,0.0,0.023392,0.0,0.0,42.75,...,-0.342791,0.0,0.0,0.0,-0.005614,0.062937,0.27574,0.0,0.0,0.0
4,AAB0162,2010-02-01,5,0.208333,0.009259,0.0,0.023148,0.0,0.5,43.20,...,-0.628322,0.0,0.0,0.0,0.000000,0.062937,0.27574,0.0,0.0,0.0


In [41]:
# ── Validation no-leakage ──
sample_user = feat['user'].iloc[0]
user_data   = feat[feat['user'] == sample_user].sort_values('week_start')

print(f'Utilisateur {sample_user} — premières semaines :')
STATIC_ZSCORE = {'ldap_zscore_departure'}
zscore_cols_check = [c for c in feat.columns if 'zscore' in c.lower() and 'max' not in c.lower() and c not in STATIC_ZSCORE]
print(user_data[['week_start','week_number'] + zscore_cols_check[:4]].head(6))

for zc in zscore_cols_check:
    vals = user_data[zc].iloc[:MIN_HIST_WEEKS]
    assert (vals == 0).all(), f'FUITE détectée : {zc} non nul semaine 1-{MIN_HIST_WEEKS}'

print(f'\n✓ No-leakage validé ({len(zscore_cols_check)} z-scores, {MIN_HIST_WEEKS} premières semaines = 0)')

expected_v3 = [
    'email_external_ratio','email_suspicious_content_ratio','email_bcc_email_ratio',
    'email_external_email_with_attachment_ratio','email_after_hours_or_weekend_ratio',
    'email_max_zscore_emails','email_avg_emails_per_week','email_avg_size_sent_email',
    'email_zscore_last_month',
    'file_copy_ratio','file_write_ratio','file_delete_ratio',
    'file_copy_to_removable_ratio','file_from_removable_ratio',
    'file_events_per_file','file_off_hours_ratio','file_suspicious_file_content_ratio',
    'file_open_then_copy_ratio','file_copy_then_delete_ratio',
    'file_max_zscore_file_activity','file_zscore_last_month',
    'logon_average_session_duration','logon_session_duration_variability',
    'logon_ratio_short_sessions','logon_used_multi_pc_simultaneously',
    'logon_unclosed_session_ratio','logon_after_hours_logon_ratio',
    'logon_max_zscore_logon','logon_max_zscore_logon_after_hours',
    'logon_zscore_last_month',
    'decoy_file_nb_access_decoy_file','psycho_ocean_risk_score',
    'device_con_duration_avg','device_con_duration_std',
    'device_ratio_short_device_conn_interval','device_avg_device_conn_per_week',
    'device_max_zscore_device_week','device_ratio_device_conn_after_hours',
    'device_zscore_last_month','ldap_zscore_departure',
    'http_avg_requests_per_day','http_total_keyword_flags',
    'http_upload_ratio','http_download_ratio','http_off_hours_ratio',
    'http_last_month_zscore','http_max_zscore_activity',
    'http_suspicious_ratio','http_unique_domains',
    'users_post_departure_activity','users_pre_departure_ratio',
    'users_activity_not_main_pc_ratio',
]
missing = [c for c in expected_v3 if c not in feat.columns]
if missing:
    print(f'⚠ Colonnes manquantes : {missing}')
else:
    print(f'✓ Toutes les {len(expected_v3)} features de features_v3 sont présentes')

all_zscores = [c for c in feat.columns if c.startswith('z_') or 'zscore' in c.lower()]
print(f'\n{len(all_zscores)} z-scores au total')
print(feat[all_zscores[:6]].describe().round(2))

Utilisateur AAB0162 — premières semaines :
  week_start  week_number  email_zscore_last_month  file_zscore_last_month  \
0 2010-01-04            1                 0.000000                    0.00   
1 2010-01-11            2                 0.000000                    0.00   
2 2010-01-18            3                 0.000000                    0.00   
3 2010-01-25            4                -9.000000                   -1.00   
4 2010-02-01            5                 0.500000                   -0.75   
5 2010-02-08            6                 0.447214                   -0.60   

   logon_zscore_last_month  device_zscore_last_month  
0                      0.0                       0.0  
1                      0.0                       0.0  
2                      0.0                       0.0  
3                      0.0                       0.0  
4                      0.0                       0.0  
5                      0.0                       0.0  

✓ No-leakage validé (5 z

In [42]:
# ── Sauvegarde ──
feat.to_parquet(OUTPUT_PATH, index=False)
print(f'Sauvegardé : {OUTPUT_PATH}')
print(f'Dimensions : {feat.shape[0]:,} lignes × {feat.shape[1]} colonnes')
print(f'Semaines   : {feat["week_start"].nunique()} ({feat["week_start"].min().date()} → {feat["week_start"].max().date()})')
print(f'Utilisateurs : {feat["user"].nunique()}')
print(f'\nColonnes :\n{feat.columns.tolist()}')

# Export CSV optionnel
feat.to_csv('features_weekly.csv', index=False)

Sauvegardé : features_weekly.parquet
Dimensions : 284,061 lignes × 86 colonnes
Semaines   : 74 (2010-01-04 → 2011-05-30)
Utilisateurs : 4000

Colonnes :
['user', 'week_start', 'week_number', 'email_external_ratio', 'email_suspicious_content_ratio', 'email_bcc_email_ratio', 'email_external_email_with_attachment_ratio', 'email_after_hours_or_weekend_ratio', 'email_max_zscore_emails', 'email_avg_emails_per_week', 'email_avg_size_sent_email', 'email_zscore_last_month', 'z_email_external_ratio', 'z_email_suspicious_content_ratio', 'z_email_bcc_email_ratio', 'z_email_external_email_with_attachment_ratio', 'z_email_after_hours_or_weekend_ratio', 'z_email_avg_emails_per_week', 'z_email_avg_size_sent_email', 'file_copy_ratio', 'file_write_ratio', 'file_delete_ratio', 'file_copy_to_removable_ratio', 'file_from_removable_ratio', 'file_events_per_file', 'file_off_hours_ratio', 'file_suspicious_file_content_ratio', 'file_open_then_copy_ratio', 'file_copy_then_delete_ratio', 'file_max_zscore_file_ac

In [43]:
feat.head()

,user,week_start,week_number,email_external_ratio,email_suspicious_content_ratio,email_bcc_email_ratio,email_external_email_with_attachment_ratio,email_after_hours_or_weekend_ratio,email_max_zscore_emails,email_avg_emails_per_week,...,z_http_avg_requests_per_day,z_http_upload_ratio,z_http_download_ratio,z_http_off_hours_ratio,z_http_suspicious_ratio,psycho_ocean_risk_score,ldap_zscore_departure,users_activity_not_main_pc_ratio,users_post_departure_activity,users_pre_departure_ratio
0,AAB0162,2010-01-04,1,0.222222,0.000000,0.0,0.000000,0.0,0.0,45.00,...,0.000000,0.0,0.0,0.0,0.000000,0.062937,0.27574,0.0,0.0,0.0
1,AAB0162,2010-01-11,2,0.166667,0.000000,0.0,0.011111,0.0,0.0,45.00,...,0.000000,0.0,0.0,0.0,0.000000,0.062937,0.27574,0.0,0.0,0.0
2,AAB0162,2010-01-18,3,0.229630,0.000000,0.0,0.022222,0.0,0.0,45.00,...,0.000000,0.0,0.0,0.0,0.000000,0.062937,0.27574,0.0,0.0,0.0
3,AAB0162,2010-01-25,4,0.216374,0.000000,0.0,0.023392,0.0,0.0,42.75,...,-0.342791,0.0,0.0,0.0,-0.005614,0.062937,0.27574,0.0,0.0,0.0
4,AAB0162,2010-02-01,5,0.208333,0.009259,0.0,0.023148,0.0,0.5,43.20,...,-0.628322,0.0,0.0,0.0,0.000000,0.062937,0.27574,0.0,0.0,0.0


In [44]:
feat.columns

Index(['user', 'week_start', 'week_number', 'email_external_ratio',
       'email_suspicious_content_ratio', 'email_bcc_email_ratio',
       'email_external_email_with_attachment_ratio',
       'email_after_hours_or_weekend_ratio', 'email_max_zscore_emails',
       'email_avg_emails_per_week', 'email_avg_size_sent_email',
       'email_zscore_last_month', 'z_email_external_ratio',
       'z_email_suspicious_content_ratio', 'z_email_bcc_email_ratio',
       'z_email_external_email_with_attachment_ratio',
       'z_email_after_hours_or_weekend_ratio', 'z_email_avg_emails_per_week',
       'z_email_avg_size_sent_email', 'file_copy_ratio', 'file_write_ratio',
       'file_delete_ratio', 'file_copy_to_removable_ratio',
       'file_from_removable_ratio', 'file_events_per_file',
       'file_off_hours_ratio', 'file_suspicious_file_content_ratio',
       'file_open_then_copy_ratio', 'file_copy_then_delete_ratio',
       'file_max_zscore_file_activity', 'file_zscore_last_month',
       'decoy_

In [46]:
feat['week_number'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51,
       52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68,
       69, 70, 71, 72, 73, 74])